# Crawl Kapruka
#### [www.kapruka.com](https://www.kapruka.com/)

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv

import asyncio

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(
        asyncio.WindowsProactorEventLoopPolicy()
    )

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

print(f"📁 Project root: {project_root}")

In [ ]:
from infrastructure.config import ( CRAWL_OUT_DIR, MARKDOWN_DIR )

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Output directories ready:")
print(f"   - Markdown: {MARKDOWN_DIR}")
print(f"   - JSONL: {CRAWL_OUT_DIR}")

In [ ]:
from services.ingest_services import KaprukaWebCrawler

print('✅ KaprukaWebCrawler loaded from service layer')

## Crawling Configurations

In [ ]:
BASE_URL = "https://www.kapruka.com"

START_PATHS = [
    "/", "/shops", "/shop" "/globalshop", "/money", "/realestate", "/contactUs", "blog-2/", "/international_gifts", "/online", "/online/cakes" "/online/combogifts", "/online/chocolates", "/online/clothing", "/online/electronics", "/online/flowers", "/online/fruitbaskets", "/online/vegetables", "/online/giftvouchers", "/online/giftset", "/online/grocery", "/online/greetingcards", "/online/hampers", "/online/Jewellery", "/online/customizedGifts", "/online/perfumes", "/online/fashion", "/online/cosmetics", "/online/schoolpride", "/online/childrens", "/online/books", "/online/pharmacy", "/online/softtoy", "/online/bicycles", "/online/toys", "/online/baby", "/online/sports", "/online/baby", "/online/home_lifestyle", "/online/pirikara", "/online/Automobile", "/online/pet", "/online/Intimate_Essentials", "/online/exports",  "/online/electronics", "/online/samedaydelivery", "/online/promotions", "shops/specialGifts/", "/online/services", "/Sri_Lanka/mobile_reloads", "shop/faq",  
]

START_URLS = [BASE_URL + path for path in START_PATHS]

EXCLUDE_PATTERNS = [
    "/login", "/terms", "/privacy", "/admin",
    "/images/", "/downloads/", "/media/"
]

MAX_DEPTH = 8
REQUEST_DELAY = 2.0
JSONL_PATH = CRAWL_OUT_DIR / "kapruka_docs.jsonl"

print(f"🌐 Crawl config:")
print(f"   - Start URLs: {len(START_URLS)}")
print(f"   - Max depth: {MAX_DEPTH}")
print(f"   - Request delay: {REQUEST_DELAY}s")

## Execute Crawling

In [ ]:
import threading
import asyncio
import sys
import time

start_time = time.time()

# Initialize crawler service
crawler = KaprukaWebCrawler(
    base_url=BASE_URL,
    max_depth=MAX_DEPTH,
    exclude_patterns=EXCLUDE_PATTERNS
)

result = {}
error = {}

def run_crawler_in_thread():
    loop = None

    try:
        if sys.platform.startswith("win"):
            asyncio.set_event_loop_policy(
                asyncio.WindowsProactorEventLoopPolicy()
            )

        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        result["documents"] = loop.run_until_complete(
            crawler.crawl_async(
                START_URLS,
                request_delay=REQUEST_DELAY
            )
        )

    except Exception as e:
        error["exception"] = e

    finally:
        if loop is not None:
            loop.close()

print(f"\n🚀 Starting crawl at {time.strftime('%H:%M:%S')}\n")

thread = threading.Thread(target=run_crawler_in_thread)
thread.start()
thread.join()

if "exception" in error:
    raise error["exception"]

documents = result["documents"]

elapsed = time.time() - start_time

print(f"\n✅ Crawl complete in {elapsed:.1f}s")
print(f"📄 Documents collected: {len(documents)}")
print(f"🔗 URLs visited: {len(crawler.visited)}")

In [ ]:
# Cell 7: Save Outputs
# Save markdown files
for i, doc in enumerate(documents):
    url_path = urlparse(doc['url']).path.strip('/').replace('/', '_')
    if not url_path:
        url_path = "homepage"
    filename = f"{i:03d}_{url_path}.md"
    
    md_file = MARKDOWN_DIR / filename
    with open(md_file, 'w', encoding='utf-8') as f:
        f.write(f"# {doc['title']}\n\n")
        f.write(f"**URL**: {doc['url']}\n\n")
        f.write(f"**Depth**: {doc['depth_level']}\n\n")
        f.write("---\n\n")
        f.write(doc['content'])

print(f"✅ Saved {len(documents)} markdown files to {MARKDOWN_DIR}")

# Save JSONL
with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"✅ Saved JSONL corpus to {JSONL_PATH}")